### Contexto del problema a resolver:

La Clínica San Gabriel llevaba el control de sus citas en cuadernos, lo que producía tres problemas: se programaban **dos pacientes con el mismo médico a la misma hora**, el **historial clínico se perdía** y **no existían reportes**.

El sistema resuelve los tres modelando cada elemento del proceso como un objeto: `Paciente`, `Medico`, `Cita`, `Historial` y `Clinica`, todas construidas sobre la clase base `Persona`.


---

# Parte 1 — Anaya Yarleque, Nahomi Alessandra

Las clases base del sistema. Aquí se definen los datos que comparten todas las personas y la estructura que guarda los antecedentes médicos.

## 1.1 Clase `Historial`

Guarda las atenciones de un paciente. Cada atención es un **diccionario**, para poder convertirlas después en una tabla de pandas sin trabajo adicional.

Se define primero porque `Paciente` la necesita: al crear un paciente se le crea también su historial.

In [ ]:
# ------------------------------------------------------------
#  Clase Historial
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Guarda las atenciones de UN paciente. Cada atencion se guarda
# como un diccionario, para poder convertirlas despues en una
# tabla de pandas sin trabajo adicional.


class Historial:
    # Atributo de CLASE: el mismo para todos los historiales
    centro = "Clinica San Gabriel"

    def __init__(self, codigo_paciente):
        # Atributos de INSTANCIA: distintos en cada objeto
        self.codigo_paciente = codigo_paciente
        self.atenciones = []

    def agregar_atencion(self, fecha, especialidad, medico, diagnostico, tratamiento):
        atencion = {
            "fecha": fecha,
            "especialidad": especialidad,
            "medico": medico,
            "diagnostico": diagnostico,
            "tratamiento": tratamiento
        }
        self.atenciones.append(atencion)

    def total_atenciones(self):
        return len(self.atenciones)

    def ultimo_diagnostico(self):
        if len(self.atenciones) == 0:
            return "Sin atenciones registradas"
        ultima = self.atenciones[-1]
        return ultima["diagnostico"]

    def mostrar(self):
        print("HISTORIAL CLINICO DEL PACIENTE " + self.codigo_paciente)
        if len(self.atenciones) == 0:
            print("   (Sin atenciones registradas)")
        else:
            numero = 1
            for a in self.atenciones:
                print("   " + str(numero) + ". " + a["fecha"] + " | " + a["especialidad"] +
                      " | Dx: " + a["diagnostico"] + " | Tto: " + a["tratamiento"])
                numero = numero + 1
        print("   Total de atenciones: " + str(self.total_atenciones()))


print("Clase Historial creada.")

Clase Historial creada.


# CLASE BASE : Persona

Reúne los datos que comparten pacientes y médicos. Todos sus atributos son privados y se acceden mediante propiedades que validan la información antes de guardarla (**encapsulamiento**).

In [ ]:
# ============================================================
#  BLOQUE 3 - CLASE BASE Persona
# ============================================================
# Persona reune los datos que comparten pacientes y medicos.
# De ella heredaran Paciente y Medico (HERENCIA), asi no se
# repite el mismo codigo en dos clases distintas.


class Persona:
    """Clase base del sistema. Guarda los datos personales.

    Todos los atributos son privados (doble guion bajo) y se
    acceden mediante propiedades con validacion, que es la forma
    correcta de aplicar ENCAPSULAMIENTO en Python.
    """

    def __init__(self, nombre, apellido, dni, telefono):
        # Se asigna con self.nombre (no self.__nombre) para que
        # se ejecute el setter y los datos entren ya validados.
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni
        self.telefono = telefono

    # ---------------- Propiedades con validacion ----------------
    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        self.__nombre = Validador.texto(valor, "nombre")

    @property
    def apellido(self):
        return self.__apellido

    @apellido.setter
    def apellido(self, valor):
        self.__apellido = Validador.texto(valor, "apellido")

    @property
    def dni(self):
        return self.__dni

    @dni.setter
    def dni(self, valor):
        self.__dni = Validador.dni(valor)

    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        self.__telefono = Validador.telefono(valor)

    @property
    def nombre_completo(self):
        """Propiedad calculada: no se guarda, se arma cuando se pide."""
        return f"{self.__nombre} {self.__apellido}"

    # ---------------- Metodos ----------------
    def mostrar_informacion(self):
        """Muestra los datos generales.

        Las clases hijas van a REDEFINIR este metodo con sus
        propios datos: eso es POLIMORFISMO.
        """
        return (f"[PERSONA]  {self.nombre_completo} | DNI: {self.__dni} | "
                f"Telefono: {self.__telefono}")

    def __str__(self):
        return self.nombre_completo


print("Bloque 3 cargado: clase base Persona lista.")

# CLASE Paciente

In [ ]:
# ============================================================
#  BLOQUE 4 - CLASE Paciente (hereda de Persona)
# ============================================================


class Paciente(Persona):
    """Paciente de la clinica.

    HERENCIA    : reutiliza nombre, apellido, dni y telefono de Persona.
    COMPOSICION : al crearse, el paciente genera su propio HistorialMedico.
    POLIMORFISMO: redefine el metodo mostrar_informacion().
    """

    def __init__(self, codigo, nombre, apellido, dni, telefono, fecha_nacimiento, tiene_seguro=False):
        # super() llama al constructor de la clase padre y evita
        # volver a escribir las validaciones de los datos personales.
        super().__init__(nombre, apellido, dni, telefono)
        self.__codigo = codigo
        self.__fecha_nacimiento = Validador.fecha(fecha_nacimiento)
        self.__tiene_seguro = bool(tiene_seguro)
        self.__historial = HistorialMedico(codigo)   # objeto dentro de otro objeto

        # Regla: la fecha de nacimiento no puede ser futura.
        if self.__fecha_nacimiento > date.today():
            raise ValueError("La fecha de nacimiento no puede ser posterior a hoy.")

    # ---------------- Propiedades ----------------
    @property
    def codigo(self):
        return self.__codigo

    @property
    def fecha_nacimiento(self):
        return self.__fecha_nacimiento

    @property
    def tiene_seguro(self):
        return self.__tiene_seguro

    @tiene_seguro.setter
    def tiene_seguro(self, valor):
        self.__tiene_seguro = bool(valor)

    @property
    def historial(self):
        return self.__historial

    # ---------------- Metodos propios ----------------
    def edad(self):
        """Calcula la edad exacta a partir de la fecha de nacimiento."""
        hoy = date.today()
        anios = hoy.year - self.__fecha_nacimiento.year
        # Si todavia no cumplio anios este ano, se resta uno.
        if (hoy.month, hoy.day) < (self.__fecha_nacimiento.month, self.__fecha_nacimiento.day):
            anios -= 1
        return anios

    def es_adulto_mayor(self):
        """Regla de negocio: los mayores de 60 anios tienen atencion preferente."""
        return self.edad() >= 60

    def registrar_atencion(self, registro):
        """Guarda una atencion en el historial del paciente."""
        self.__historial.agregar_registro(registro)

    # ---------------- Polimorfismo ----------------
    def mostrar_informacion(self):
        """Redefine el metodo del padre agregando los datos del paciente."""
        seguro = "SI" if self.__tiene_seguro else "NO"
        preferente = "  (*) Atencion preferente" if self.es_adulto_mayor() else ""
        return (f"[PACIENTE] {self.__codigo} | {self.nombre_completo} | DNI: {self.dni} | "
                f"Edad: {self.edad()} anios | Seguro: {seguro}{preferente}")

    def __str__(self):
        return f"{self.__codigo} - {self.nombre_completo}"


print("Bloque 4 cargado: clase Paciente lista.")